In [1]:
import pandas as pd

In [2]:
final_data = pd.read_parquet("../data/processed/final_data.parquet")

In [28]:
from m5_forecasting.features.build_features import create_time_features
from m5_forecasting.features.build_features import (
    create_time_features,
    create_lag_features
)
from m5_forecasting.features.build_features import create_rolling_features
from m5_forecasting.features.encoding import encode_categorical_features
from m5_forecasting.features.scaling import scale_features
from m5_forecasting.data.sequence import create_sequences
from m5_forecasting.data.split import train_val_test_split

In [4]:
final_data = create_time_features(final_data)

In [5]:
final_data = create_time_features(final_data)
final_data = create_lag_features(final_data)
final_data = create_rolling_features(final_data)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,sell_price,day,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,NaN,29,5,4,1,NaN,NaN,NaN,NaN,NaN
100,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0,2011-01-30,11101,...,NaN,30,6,4,1,0.0,NaN,NaN,NaN,NaN
200,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0,2011-01-31,11101,...,NaN,31,0,5,0,0.0,NaN,NaN,NaN,NaN
300,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_4,0,2011-02-01,11101,...,NaN,1,1,5,0,0.0,NaN,NaN,NaN,NaN
400,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_5,0,2011-02-02,11101,...,NaN,2,2,5,0,0.0,NaN,NaN,NaN,NaN


In [22]:
product = final_data["item_id"].iloc[0]
store = final_data["store_id"].iloc[0]

sample = final_data[
    (final_data["item_id"] == product) &
    (final_data["store_id"] == store)
]

sample[
    ["date", "sales", "lag_1", "lag_7", "lag_28"]
].head(35)

,date,sales,lag_1,lag_7,lag_28
0,2011-01-29,0,NaN,NaN,NaN
100,2011-01-30,0,0.0,NaN,NaN
200,2011-01-31,0,0.0,NaN,NaN
300,2011-02-01,0,0.0,NaN,NaN
400,2011-02-02,0,0.0,NaN,NaN
500,2011-02-03,0,0.0,NaN,NaN
600,2011-02-04,0,0.0,NaN,NaN
700,2011-02-05,0,0.0,0.0,NaN
800,2011-02-06,0,0.0,0.0,NaN
900,2011-02-07,0,0.0,0.0,NaN


In [23]:
final_data, encoders = encode_categorical_features(final_data)

In [12]:
final_data, scaler = scale_features(final_data)

In [24]:
feature_columns = [
    "sell_price",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "is_weekend",
]

In [25]:
target_column = "sales"

In [27]:
X, y = create_sequences(
    final_data,
    feature_columns,
    target_column,
    sequence_length=28
)

In [18]:
final_data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,sell_price,day,day_of_week,week_of_year,is_weekend,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,d_1,0,2011-01-29,11101,...,NaN,0.933333,0.833333,0.057692,1.0,NaN,NaN,NaN,NaN,NaN
100,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,d_2,0,2011-01-30,11101,...,NaN,0.966667,1.000000,0.057692,1.0,0.0,NaN,NaN,NaN,NaN
200,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,d_3,0,2011-01-31,11101,...,NaN,1.000000,0.000000,0.076923,0.0,0.0,NaN,NaN,NaN,NaN
300,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,d_4,0,2011-02-01,11101,...,NaN,0.000000,0.166667,0.076923,0.0,0.0,NaN,NaN,NaN,NaN
400,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,d_5,0,2011-02-02,11101,...,NaN,0.033333,0.333333,0.076923,0.0,0.0,NaN,NaN,NaN,NaN


In [29]:
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
) = train_val_test_split(X, y)

In [30]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (133890, 28, 11)
Validation: (28691, 28, 11)
Testing: (28691, 28, 11)
